In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import random
import torch
import torch.nn as nn

from ucimlrepo import fetch_ucirepo 
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

### Data Fetching and Processing

In [5]:
try:
    default_of_credit_card_clients = fetch_ucirepo(id=350)
    df = dataset.data.original
    print("Sucessfully fetced dataset from ucirepo.")
except Exception as e:
    df = pd.read_csv('default_of_credit_card_clients.csv')
    print(f"Ucirepo failed: {e}. Uploaded data from local environment.")

Ucirepo failed: Error connecting to server. Uploaded data from local environment.


In [6]:
df.columns = df.iloc[0]
df = df.iloc[1:].copy()
df = df.rename(columns={"default payment next month": "default"})
df = df.drop(columns=["ID"])

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.reset_index(drop=True)

print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

Rows: 30,000   Columns: 24
Missing values: 0


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


### Data Exploration

In [ ]:
counts = df["default"].value_counts()
pcts   = df["default"].value_counts(normalize=True) * 100

print("Class Distribution")
print(f"  No Default (0) : {counts[0]:,}  ({pcts[0]:.2f}%)")
print(f"  Default    (1) : {counts[1]:,}  ({pcts[1]:.2f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["No Default (0)", "Default (1)"], counts,
              color=["#4CAF50", "#F44336"], edgecolor="black", width=0.5)
for bar, count, pct in zip(bars, counts, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 300,
            f"{count:,}\n({pct:.1f}%)",
            ha="center", va="bottom", fontsize=10)
ax.set_title("Class Distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("Count")
ax.set_ylim(0, counts.max() * 1.15)
# sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Default rate by PAY_0
pay_dr = df.groupby("PAY_0")["default"].mean() * 100
axes[0].bar(pay_dr.index.astype(str), pay_dr.values,
            color="#FF9800", edgecolor="black")
axes[0].set_title("Default Rate by Payment Status (PAY_0)", fontweight="bold")
axes[0].set_xlabel("PAY_0  (-2=no use, -1=on time, 1–8=months delayed)")
axes[0].set_ylabel("Default Rate (%)")

# Default rate by EDUCATION
edu_labels = {1: "Graduate", 2: "University", 3: "High School", 4: "Others"}
edu_dr = df.groupby("EDUCATION")["default"].mean() * 100
edu_dr.index = [edu_labels.get(i, str(i)) for i in edu_dr.index]
axes[1].bar(edu_dr.index, edu_dr.values, color="#2196F3", edgecolor="black")
axes[1].set_title("Default Rate by Education Level", fontweight="bold")
axes[1].set_ylabel("Default Rate (%)")
axes[1].tick_params(axis="x", rotation=15)

# Credit limit distribution by class
df[df["default"] == 0]["LIMIT_BAL"].plot(
    kind="hist", bins=40, alpha=0.6, ax=axes[2], color="#4CAF50", label="No Default")
df[df["default"] == 1]["LIMIT_BAL"].plot(
    kind="hist", bins=40, alpha=0.6, ax=axes[2], color="#F44336", label="Default")
axes[2].set_title("Credit Limit Distribution by Class", fontweight="bold")
axes[2].set_xlabel("Credit Limit (LIMIT_BAL)")
axes[2].set_ylabel("Frequency")
axes[2].legend()

plt.suptitle("Exploratory Data Analysis", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Small Language Model (SLM)

#### Defining Feature Groups

In [7]:
# These columns represent STATIC (non-temporal) customer information.
# They do not change over time, so they will be grouped into ONE token.
# This token captures demographic and financial profile of the customer.
STATIC_COLS = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]

# These columns represent TEMPORAL behaviour across months.
# Each set of PAY, BILL_AMT, PAY_AMT corresponds to a specific month.

# Repayment status (ordinal feature):
# -1 = paid on time, 1 = delay, ..., 9 = severe delay
# These values will later be embedded to capture behavioural patterns.
PAY_COLS = ["PAY_6", "PAY_5", "PAY_4", "PAY_3", "PAY_2", "PAY_0"]

# Monthly bill amounts (continuous values)
# These reflect how much the customer was billed each month.
BILL_COLS = ["BILL_AMT6", "BILL_AMT5", "BILL_AMT4", "BILL_AMT3", "BILL_AMT2", "BILL_AMT1"]

# Monthly payment amounts (continuous values)
# These reflect how much the customer actually paid each month.
AMT_COLS = ["PAY_AMT6", "PAY_AMT5", "PAY_AMT4", "PAY_AMT3", "PAY_AMT2", "PAY_AMT1"]

# Target variable:
# 1 = default, 0 = no default
TARGET_COL = "default"

# Feature type separation (important for embedding design)

# Static numerical features:
# These will be scaled and projected using a linear layer.
NUMERIC_STATIC_COLS = ["LIMIT_BAL", "AGE"]

# Monthly numerical features:
# These include both financial amounts and repayment status (treated as ordinal).
# They will be normalised before being converted into embeddings.
MONTHLY_NUMERIC_COLS = BILL_COLS + AMT_COLS + PAY_COLS

### Train-Test Split

In [8]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[TARGET_COL]
)

print(train_df.shape, test_df.shape)

(24000, 24) (6000, 24)


### Scaling Numerical Columns

In [ ]:
scaler_static = StandardScaler()
scaler_monthly = StandardScaler()

train_df = train_df.copy()
test_df = test_df.copy()

train_df[NUMERIC_STATIC_COLS] = scaler_static.fit_transform(train_df[NUMERIC_STATIC_COLS])
test_df[NUMERIC_STATIC_COLS] = scaler_static.transform(test_df[NUMERIC_STATIC_COLS])

train_df[MONTHLY_NUMERIC_COLS] = scaler_monthly.fit_transform(train_df[MONTHLY_NUMERIC_COLS])
test_df[MONTHLY_NUMERIC_COLS] = scaler_monthly.transform(test_df[MONTHLY_NUMERIC_COLS])

### Dataset Construction and Token Preparation
The dataset class converts each record into structured inputs for the transformer. Static features are grouped into a single token, while monthly financial behaviour is represented as a sequence of six temporal tokens. This preserves the temporal structure of the data and enables the application of self-attention mechanisms. Repayment status values are adjusted for compatibility with embedding layers.

In [ ]:
class CreditDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        static_num = torch.tensor(
            [row["LIMIT_BAL"], row["AGE"]],
            dtype=torch.float32
        )

        static_cat = torch.tensor(
            [int(row["SEX"]), int(row["EDUCATION"]), int(row["MARRIAGE"])],
            dtype=torch.long
        )

        monthly_num = []
        monthly_pay = []

        for p, b, a in zip(PAY_COLS, BILL_COLS, AMT_COLS):
            monthly_num.append([row[b], row[a]])
            monthly_pay.append(int(row[p]) + 2)  # shift to non-negative indices

        monthly_num = torch.tensor(monthly_num, dtype=torch.float32)   # (6, 2)
        monthly_pay = torch.tensor(monthly_pay, dtype=torch.long)      # (6,)
        target = torch.tensor(row[TARGET_COL], dtype=torch.float32)

        return {
            "static_num": static_num,
            "static_cat": static_cat,
            "monthly_num": monthly_num,
            "monthly_pay": monthly_pay,
            "target": target
        }

In [ ]:
train_dataset = CreditDataset(train_df)
test_dataset = CreditDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

### Batch Structure Verification
A sample batch is extracted from the DataLoader to verify the structure of the input data. This step ensures that the dataset has been correctly organised into static and temporal components. The output confirms that each batch contains static numerical and categorical features, as well as temporal features represented over six time steps. This structured format is essential for constructing token embeddings and feeding the data into the transformer model.

In [ ]:
batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, v.shape)

### Positional Encoding
Positional encoding is used to introduce information about the order of tokens in the sequence, as transformer models do not inherently capture positional relationships. A sinusoidal encoding scheme is applied, where each position in the sequence is mapped to a unique vector using sine and cosine functions of different frequencies. These positional vectors are added to the token embeddings, allowing the model to distinguish between different time steps (e.g., April to September) and learn temporal dependencies through self-attention

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=20):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

### Token Embedding Layer
This layer transforms tabular data into token embeddings for the transformer. Static features are combined into a single token using numerical projections and categorical embeddings, while monthly financial behaviour is represented as a sequence of tokens incorporating numerical values, repayment status embeddings, and month identifiers. The resulting sequence enables the transformer to process structured data using self-attention.

In [ ]:
class CreditTokenEmbedding(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.sex_emb = nn.Embedding(4, d_model)
        self.education_emb = nn.Embedding(8, d_model)
        self.marriage_emb = nn.Embedding(5, d_model)

        self.static_num_proj = nn.Linear(2, d_model)
        self.static_fusion = nn.Linear(d_model, d_model)

        self.monthly_num_proj = nn.Linear(2, d_model)
        self.pay_status_emb = nn.Embedding(16, d_model)
        self.month_emb = nn.Embedding(6, d_model)

        self.dropout = nn.Dropout(0.1)

    def forward(self, static_num, static_cat, monthly_num, monthly_pay):
        batch_size = static_num.size(0)

        static_num_vec = self.static_num_proj(static_num)
        sex_vec = self.sex_emb(static_cat[:, 0])
        edu_vec = self.education_emb(static_cat[:, 1])
        marriage_vec = self.marriage_emb(static_cat[:, 2])

        static_token = static_num_vec + sex_vec + edu_vec + marriage_vec
        static_token = self.static_fusion(static_token).unsqueeze(1)  # (batch, 1, d_model)

        monthly_num_vec = self.monthly_num_proj(monthly_num)
        pay_vec = self.pay_status_emb(monthly_pay)

        month_idx = torch.arange(6, device=monthly_num.device).unsqueeze(0).expand(batch_size, 6)
        month_vec = self.month_emb(month_idx)

        monthly_tokens = monthly_num_vec + pay_vec + month_vec  # (batch, 6, d_model)

        tokens = torch.cat([static_token, monthly_tokens], dim=1)  # (batch, 7, d_model)
        return self.dropout(tokens)

### Multi Head Self-Attention Mechanism
The input embeddings are linearly transformed into queries (Q), keys (K), and values (V), which are used to compute attention scores. These scores determine how much each token attends to others in the sequence.

The attention operation is performed across multiple heads, allowing the model to capture different types of relationships simultaneously. The outputs from all heads are then concatenated and projected back into the original embedding space. This mechanism enables the model to capture dependencies across time steps, such as how past repayment behaviour influences future default risk.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)

        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        out = self.W_o(context)

        return out, attn_weights

### Transformer Block Architecture
Transformer block composed of multi-head self-attention and a position-wise feed-forward network. Residual connections and layer normalisation are applied after each sub-layer to improve training stability and convergence, while dropout provides regularisation. Stacking such blocks enables the model to learn increasingly complex relationships across the input sequence.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, attn_weights = self.attn(x)
        x = self.norm1(x + self.dropout1(attn_out))

        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout2(ff_out))

        return x, attn_weights

### Transformer-Based Classification Model
This model integrates all components to perform credit default prediction using a transformer architecture. Input data are first converted into token embeddings,representing both static and temporal features, and enhanced with positional encoding to preserve the order of the sequence.

The embedded sequence is then processed through multiple stacked transformer blocks, each consisting of multi-head self-attention and feed-forward layers, enabling the model to capture complex relationships across time steps.

The final sequence representation is aggregated using mean pooling to produce a fixed-length vector, which is passed through a linear classification layer to predict the probability of default. Additionally, attention weights are retained for interpretability, allowing analysis of which time steps contribute most to the prediction.

In [ ]:
class CreditTransformer(nn.Module):
    def __init__(self, d_model=64, num_heads=4, ff_dim=128, num_layers=2, dropout=0.1):
        super().__init__()

        self.token_embedding = CreditTokenEmbedding(d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len=10)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.classifier = nn.Linear(d_model, 1)

    def forward(self, static_num, static_cat, monthly_num, monthly_pay):
        x = self.token_embedding(static_num, static_cat, monthly_num, monthly_pay)
        x = self.positional_encoding(x)

        attention_maps = []
        for block in self.blocks:
            x, attn = block(x)
            attention_maps.append(attn)

        pooled = x.mean(dim=1)
        logits = self.classifier(pooled).squeeze(-1)

        return logits, attention_maps

### Initialize

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
model = CreditTransformer(
    d_model=64,
    num_heads=4,
    ff_dim=128,
    num_layers=2,
    dropout=0.1
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### Model Training

In [ ]:
# SLM Model here

# Benchmark - Random Forest

In [9]:
X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

In [18]:
param_dist = {
    "n_estimators"     : [100, 200, 300, 500, 700, 1000],
    "max_depth"        : [None, 10, 20, 30, 50, 100],
    "min_samples_split": [2, 5, 10, 15, 20, 30],
    "min_samples_leaf" : [1, 2, 4, 8],
    "max_features"     : ["sqrt", "log2"],
}

#cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

randomized_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    param_distributions=param_dist,
    n_iter=30,
    scoring="f1_macro",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
randomized_search.fit(X_train, y_train)

print(f"   Best Parameters : {randomized_search.best_params_}")
print(f"   Best f1 Score   : {randomized_search.best_score_:.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
   Best Parameters : {'n_estimators': 200, 'min_samples_split': 30, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 100}
   Best f1 Score   : 0.7080


In [ ]:
param_grid = {
    'n_estimators': [75, 100, 125],
    'max_depth': [15, 20, 25],
    'min_samples_split': [8, 10, 12]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5, 
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_combined, y_train.values.ravel())

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score (F1): {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_

final_preds = best_rf.predict(X_test_combined)

print("--- Final GridSearchCV Random Forest Report ---")
print(classification_report(y_test, final_preds))